# Function Calling과 Tools

이 노트북은 OpenAI Responses API의 **Built-in Tools**와 **Function Calling**을 실습하는 강의자료입니다.

> 모델은 함수를 직접 실행하지 않습니다. 모델이 `function_call`을 반환하면, 우리 Python 프로그램이 함수를 실행하고 결과를 다시 모델에 전달합니다.


## 학습 목표

1. Tools와 Function Calling의 역할을 설명할 수 있습니다.
2. Web Search, File Search 같은 내장 도구를 사용할 수 있습니다.
3. 사용자 함수를 JSON Schema로 정의하고 모델의 호출 요청을 처리할 수 있습니다.
4. 여러 도구를 연결한 간단한 AI 도우미를 설계할 수 있습니다.


## 1. Function Calling과 Tools

- **Function Calling**: 모델이 개발자가 제공한 함수의 이름과 인자(JSON)를 선택하도록 하는 기능
- **Tools**: 모델이 외부 정보·기능을 사용할 수 있게 하는 더 넓은 개념
- 따라서 Function Calling은 Tools의 한 종류입니다.

| 구분 | 설명 |
|---|---|
| Function tool | 개발자가 정의한 함수. 날씨 API, 주문 조회, 사내 DB, 계산기 등에 연결 |
| Built-in tool | OpenAI가 호스팅하는 내장 도구. Web Search, File Search, Computer Use 등 |
| MCP / 외부 연결 | MCP 서버 등을 통해 외부 서비스의 도구를 연결하는 방식 |
| Agent | 여러 도구를 선택·호출하며 여러 단계의 작업을 수행하는 실행 구조 |


## 2. 왜 Tools가 필요한가?

모델은 학습된 패턴을 바탕으로 응답을 생성합니다. 기본 상태에서는 실시간 날씨, 사내 데이터베이스, 사용자의 파일, 주문 상태를 직접 확인할 수 없습니다.

도구를 연결하면 현재 날씨 확인, 데이터베이스 조회, 이메일 전송, 파일 읽기, 계산기 실행, 웹 검색, 사내 시스템 접근을 실제 데이터로 처리할 수 있습니다.


## 3. 최신 OpenAI API에서의 변화

| 이전 표현/방식 | 현재 권장 방식 |
|---|---|
| `functions`, `function_call` 파라미터 | `tools` 안에 `type: "function"` 정의 |
| Chat Completions 중심 | **Responses API** 중심 |
| `web_search_preview` | `web_search` |
| 임의 JSON 인자 처리 | JSON Schema + `strict: true` 권장 |


## 4. Tools의 공통 동작 흐름

1. 사용자가 질문을 입력합니다.
2. 모델이 의도를 이해하고 필요한 도구와 인자를 선택합니다.
3. Function tool이라면 우리 애플리케이션 서버가 실제 함수를 실행합니다.
4. Built-in tool이라면 OpenAI 플랫폼이 Web/File Search 등을 실행합니다.
5. 모델이 도구 결과를 바탕으로 최종 답변을 생성합니다.

```text
User 질문 → GPT 판단 → Function Call 생성 → 서버에서 함수 실행 → 결과를 GPT에 전달 → 최종 자연어 응답 생성
```


## 5. 실습 준비

터미널에서 필요한 패키지를 설치하고, API 키는 `.env` 파일에 저장합니다. API 키를 노트북이나 GitHub에 직접 적지 마세요.

```bash
uv add openai python-dotenv requests
```

`.env` 파일 예시:
```text
OPENAI_API_KEY=sk-...
```


In [5]:
import json
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(".env 파일에 OPENAI_API_KEY를 설정하세요.")

client = OpenAI()
default_model = "gpt-6-luna"  # 사용 가능한 모델로 변경 가능


## 6. OpenAI Built-in Tools

| 도구 | type | 주된 역할 |
|---|---|---|
| Web Search | `web_search` | 최신 웹 정보 검색 및 출처 기반 답변 |
| File Search | `file_search` | Vector Store에 등록한 문서 검색(RAG) |
| Function tool | `function` | 개발자 시스템·외부 API 연결 |
| Computer Use | 제품/환경별 설정 참조 | 웹·컴퓨터 화면 작업 자동화 |

이미지 분석은 보통 별도의 `image_analysis` 도구를 선언하지 않고, 이미지 입력을 지원하는 모델에 이미지를 함께 전달합니다.


### 6-1. Web Search

Web Search는 최신 뉴스, 시장 정보, 공식 문서처럼 최신성이 중요한 질문에 적합합니다.


In [6]:
question = "서울의 오늘 날씨를 알려줘."

response = client.responses.create(
    model=default_model,
    tools=[{"type": "web_search"}],
    input=question,
)

print(response.output_text)


서울은 현재 **맑고 12°C**예요. 


### 6-2. File Search + 문서 요약

`file_search`는 파일을 바로 읽는 방식이 아닙니다. 먼저 Vector Store에 파일을 등록하고, 그 저장소 ID를 도구에 연결합니다. `PDF_PATH`를 실제 PDF 파일 경로로 바꾸세요.


In [7]:
# 1. Vector Store 생성
vector_store = client.vector_stores.create(
    name="PDF 저장소"
)

# 2. a.pdf 업로드 + Vector Store에 등록
file_batch = client.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vector_store.id,
    files=[open(".\\data\\Hotel Service Manual.pdf", "rb")]
)

print("업로드 상태:", file_batch.status)
print("파일 개수:", file_batch.file_counts)

# 3. File Search로 PDF 검색/요약
response = client.responses.create(
    model=default_model,
    tools=[{
            "type": "file_search",
            "vector_store_ids": [vector_store.id]
        }],
    input="업로드한 PDF를 5줄로 요약해줘."
)

print(response.output_text)


업로드 상태: completed
파일 개수: FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)
- 호텔 직원은 미소와 정중한 인사로 고객을 맞이하고, 개인화된 응대로 환대를 제공한다. 
- 체크인은 15시, 체크아웃은 11시이며, 객실 상황에 따라 조기 체크인·늦은 체크아웃 또는 짐 보관을 안내한다. 
- 전 객실 무료 Wi-Fi와 180대 무료 주차장을 제공하며, 장애인 편의시설과 유니버설 룸도 안내한다. 
- 반려동물은 동반할 수 없고, 룸서비스는 23시까지 운영하며 객실은 금연이다. 취소 수수료는 전날 30%, 당일 50%, 노쇼 100%다. fileciteturn0file1turn0file2
- 결제는 현금·신용카드·직불카드 등을 지원하며, 모든 고객을 존중하고 배려하는 전문적인 태도를 강조한다. 


## 7. Function Tool 실습: 날씨 함수

먼저 실제 API 대신 **mock data**를 반환하는 함수를 만듭니다. Function Calling의 핵심은 모델이 함수를 선택하고, 결과를 다시 받아 답변하는 흐름입니다.


In [8]:
def get_weather(location: str) -> str:
    """지정한 도시의 날씨 정보를 반환합니다."""
    weather_data = {
        "서울": "맑고 기온은 24도입니다.",
        "부산": "구름이 조금 있고 기온은 22도입니다.",
        "제주": "바람이 불고 기온은 21도입니다.",
        "뉴욕": "비가 오고 있으며 기온은 15도입니다.",
        "도쿄": "흐리고 기온은 20도입니다.",
    }
    return weather_data.get(location, f"{location}의 날씨 정보가 없습니다.")


### 7-1. 함수 스키마 정의

도구 스키마는 모델에게 **어떤 함수가 있고, 언제 쓰며, 어떤 인자를 받는지** 알려주는 설명서입니다.

`strict: true`를 사용하면 `parameters`에 `additionalProperties: false`를 설정하고, 모든 속성을 `required`에 포함해야 합니다.


In [9]:
weather_tools = [{
    "type": "function",
    "name": "get_weather",
    "description": "지정한 도시의 현재 날씨를 조회한다.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "날씨를 조회할 도시 이름. 예: 서울, 부산, 제주",
            }
        },
        "required": ["location"],
        "additionalProperties": False,
    },
    "strict": True,
}]

# 모델이 생성한 함수 이름을 실제 Python 함수와 연결합니다.
tool_map = {"get_weather": get_weather}


### 7-2. 1차 요청: 모델의 도구 호출 확인

`tool_choice`의 기본값은 `auto`입니다. 모델이 도구가 필요하다고 판단하면 `function_call`을 반환하고, 필요 없으면 바로 일반 답변을 반환할 수 있습니다.


In [10]:
input_message = "서울 날씨 어때?"

response_1 = client.responses.create(
    model=default_model,
    input=input_message,
    tools=weather_tools,
    tool_choice="auto",
)

for item in response_1.output:
    print("type:", item.type)
    if item.type == "function_call":
        print("call_id:", item.call_id)
        print("name:", item.name)
        print("arguments:", item.arguments)


type: function_call
call_id: call_ttAyFpH9FQrCuoIKOhqlMRzU
name: get_weather
arguments: {"location":"서울"}


### 7-3. 함수 실행 후 결과를 모델에 전달

`function_call_output`의 `call_id`는 반드시 모델이 반환한 호출 ID와 같아야 합니다.


In [12]:
tool_outputs = []

for item in response_1.output:
    if item.type != "function_call":
        continue

    arguments = json.loads(item.arguments)
    function = tool_map[item.name]
    result = function(**arguments)

    tool_outputs.append({
        "type": "function_call_output",
        "call_id": item.call_id,
        "output": result,
    })

print(tool_outputs)


[{'type': 'function_call_output', 'call_id': 'call_ttAyFpH9FQrCuoIKOhqlMRzU', 'output': '맑고 기온은 24도입니다.'}]


In [13]:
if not tool_outputs:
    # 모델이 도구 없이 바로 답한 경우
    print(response_1.output_text)
else:
    # 도구 실행 결과를 전달하여 최종 자연어 답변을 만듭니다.
    response_2 = client.responses.create(
        model=default_model,
        previous_response_id=response_1.id,
        input=tool_outputs,
        tools=weather_tools,
    )
    print(response_2.output_text)


서울은 현재 맑고, 기온은 24도예요.


### 7-4. 재사용 가능한 Function Calling 실행 함수

여러 도구가 있거나 모델이 연속으로 도구를 호출할 수 있으므로, 실무에서는 반복 처리 구조를 만듭니다.


In [14]:
def run_with_tools(user_input: str, tools: list, tool_map: dict) -> str:
    """모델의 함수 호출 요청을 모두 처리한 뒤 최종 답변을 반환합니다."""
    response = client.responses.create(
        model=default_model, input=user_input, tools=tools, tool_choice="auto"
    )

    while True:
        tool_outputs = []
        for item in response.output:
            if item.type != "function_call":
                continue
            try:
                arguments = json.loads(item.arguments)
                result = tool_map[item.name](**arguments)
            except Exception as error:
                result = {"error": f"도구 실행 중 오류: {error}"}

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps(result, ensure_ascii=False)
                if isinstance(result, dict) else str(result),
            })

        if not tool_outputs:
            return response.output_text

        response = client.responses.create(
            model=default_model,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools,
        )

print(run_with_tools("제주 날씨를 알려줘.", weather_tools, tool_map))


현재 제주는 바람이 불고, 기온은 21도예요.


## 8. 미니 과제: 멀티 Tool AI 도우미 만들기

OpenAI Function Calling을 활용해 사용자의 질문에 맞는 함수를 선택하고, 실행 결과를 바탕으로 답변하는 AI 도우미를 만드세요.

| 함수 이름 | 기능 | 예시 질문 |
|---|---|---|
| `add` | 두 수를 더한다 | 15와 27을 더해줘 |
| `subtract` | 두 수를 뺀다 | 50에서 18을 빼줘 |
| `multiply` | 두 수를 곱한다 | 7 곱하기 8은? |
| `divide` | 두 수를 나눈다 | 100을 4로 나눠줘 |
| `get_weather` | 도시의 날씨를 알려준다 | 서울 날씨 알려줘 |
| `get_exchange_rate` | 특정 통화의 환율 정보를 알려준다 | 오늘 달러 환율 알려줘 |

필수 요구사항: 6개 함수를 `tools`에 등록하고, 0으로 나누는 경우 및 지원하지 않는 통화의 예외를 처리하세요. 모델이 함수 호출을 요청하면 `function_call_output`으로 결과를 다시 전달해야 합니다.


### 테스트 질문

- 12와 8을 더해줘.
- 100에서 35를 빼면 얼마야?
- 10을 0으로 나눠줘.
- 서울 날씨와 달러 환율을 함께 알려줘.
- 100달러를 원화로 계산해줘.


## 9. Retrieval / Knowledge Tools

Retrieval은 필요한 정보를 먼저 검색한 후 GPT가 답변을 생성하는 방식입니다. 모델의 학습 데이터만 사용하는 것이 아니라 실제 문서에서 정보를 찾아 답변하므로 기업 내부 문서 QA에 자주 사용합니다.

대표적인 대상은 PDF, Word, PowerPoint, Excel, CSV, TXT, Markdown, HTML, JSON, XML과 같은 문서입니다. Notion, Google Drive, Confluence, SharePoint, GitHub, Jira, Wiki와 SQL·NoSQL·CRM·ERP도 Retrieval의 데이터 소스가 될 수 있습니다.

OpenAI의 File Search는 이러한 Retrieval을 빠르게 시작할 수 있는 내장 도구입니다.
